# 06 — Diagnostics

PM-grade defensibility checks: IC, decile anatomy, factor neutrality, regime stability.


In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config, eval as evalmod


In [ ]:
panel = pd.read_parquet(config.PANEL_DAILY_PARQUET)
panel['date'] = pd.to_datetime(panel['date'])


## 1. IC by month for each baseline factor (rungs 1-3)


In [ ]:
for score_col in ['mom_12_1', 'ewm_h60_skip']:
    ic_series = evalmod.ic_by_month(panel, score_col, ret_fut_col='ret_fut_1m')
    summary = evalmod.ic_summary(ic_series)
    print(f'{score_col:<20} mean IC = {summary["mean_ic"]:.4f}  IR = {summary["ir"]:.3f}  hit = {summary["hit_rate"]:.2f}')


## 2. Decile anatomy (D10 - D1 spread)


In [ ]:
for score_col in ['mom_12_1', 'ewm_h60_skip']:
    d = evalmod.decile_returns(panel, score_col, ret_fut_col='ret_fut_1m')
    summary = evalmod.decile_spread_summary(d)
    print(f'{score_col:<20} D10-D1 mean = {summary["hi_lo_mean"]:.4f}  t = {summary["hi_lo_t"]:.2f}  monotonic = {summary["monotonic"]}')


## 3. FF5 + UMD neutrality regression


In [ ]:
# Requires daily strategy returns (from outputs/) and FF5 daily factors
ff5 = pd.read_parquet(config.FF5_RAW)
ff5['date'] = pd.to_datetime(ff5['date'])

RUNG_IDS = ['rung_1_simple_momentum_decile', 'rung_5_tcnn_1ch']
master = evalmod.load_master_results(RUNG_IDS)
for exp_id in RUNG_IDS:
    df = master[master['experiment_id'] == exp_id].sort_values('date').set_index('date')
    daily = df['return']
    result = evalmod.ff5_neutrality(daily, ff5, include_umd=False)  # umd typically pulled separately
    print(f'{exp_id:<35} alpha = {result["alpha_ann"]*100:.2f}% (t = {result["alpha_t"]:.2f}), R² = {result["r_squared"]:.3f}')
